# Lab 5: A Phishing E-mail Detector that Reads Text
Course 314341 AI in Cybersecurity and Cloud Computing, Ajloun National University. Dr. Mohammed Tawfik.

Run the cells from top to bottom in Google Colab (Runtime → Run all, or Shift+Enter cell by cell). Read the comments: lines that start with # explain the code.

**Data:** `Phishing_Email.csv` from Kaggle [Phishing Email Detection](https://www.kaggle.com/datasets/subhajournal/phishingemails) (about 18,650 e-mails). Download it, then upload it to Colab with the **Files** panel on the left.

## 1. Load and clean the e-mails

In [ ]:
import pandas as pd
data = pd.read_csv("Phishing_Email.csv")
print(data.shape)
print(data.columns.tolist())
data = data.dropna(subset=["Email Text"])                 # some e-mails have no text
data = data[data["Email Text"].str.strip() != ""]
print(data["Email Type"].value_counts())                  # Phishing Email / Safe Email

## 2. Split: 70% training, 30% test

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    data["Email Text"], data["Email Type"],
    test_size=0.3, stratify=data["Email Type"], random_state=42)
print(len(X_train), "training e-mails,", len(X_test), "test e-mails")

## 3. Turn words into numbers (TF-IDF)
TF-IDF gives each word a number: high when the word is frequent in this e-mail but rare in other e-mails.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(max_features=5000, stop_words="english")
Xtr = vec.fit_transform(X_train)       # learn the vocabulary from the TRAINING e-mails only
Xte = vec.transform(X_test)            # reuse it for the test e-mails (no leakage)
print(Xtr.shape)

## 4. Train and test a logistic regression model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
model = LogisticRegression(max_iter=1000)
model.fit(Xtr, y_train)
y_pred = model.predict(Xte)
print(confusion_matrix(y_test, y_pred, labels=["Phishing Email", "Safe Email"]))
print(classification_report(y_test, y_pred))

## 5. Which words push an e-mail towards phishing?

In [ ]:
import numpy as np
words = np.array(vec.get_feature_names_out())
order = np.argsort(model.coef_[0])                        # low = first class, high = second class
print("Class order:", model.classes_)
print("Words that point to", model.classes_[0], ":", words[order[:20]])
print("Words that point to", model.classes_[1], ":", words[order[-20:]])

## 6. Try your own e-mails

In [ ]:
my_emails = [
    "Your account will be suspended today. Verify your password now at http://203.0.113.45/login",
    "Hi team, the lab meeting is moved to Tuesday at 10. See you there.",
]
print(model.predict(vec.transform(my_emails)))